In [2]:

# ============================================================
# RAILPULSE AI — STEP 03: DATA VALIDATION & SPLIT PREPARATION
# ============================================================

import os
import numpy as np
import pandas as pd
from collections import defaultdict

# ------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ------------------------------------------------------------

FEATURE_PATH = "../data/processed/railpulse_model_features.csv"
OUTPUT_DIR = "../outputs/tables"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000
RANDOM_STATE = 42

FEATURE_COLS = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence",
    "arrival_hour_sin",
    "arrival_hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend"
]

TARGET_COL = "next_arr_delay"

REQUIRED_COLS = [
    "train",
    "date",
    "current_station",
    "next_station"
] + FEATURE_COLS + [TARGET_COL]

# ------------------------------------------------------------
# 2. CHECK FILE AND COLUMNS
# ------------------------------------------------------------

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(
        f"Feature dataset not found: {FEATURE_PATH}"
    )

header = pd.read_csv(FEATURE_PATH, nrows=0)
available_cols = header.columns.tolist()

missing_cols = [
    col for col in REQUIRED_COLS
    if col not in available_cols
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print("=" * 65)
print("RAILPULSE AI — DATA VALIDATION")
print("=" * 65)

print("\nDataset:", FEATURE_PATH)
print("Predictor columns:", len(FEATURE_COLS))
print("Target column:", TARGET_COL)

# ------------------------------------------------------------
# 3. INITIALIZE AUDIT VARIABLES
# ------------------------------------------------------------

total_rows = 0
all_trains = set()

missing_counts = defaultdict(int)
invalid_numeric = defaultdict(int)

feature_min = {
    col: float("inf") for col in FEATURE_COLS
}

feature_max = {
    col: float("-inf") for col in FEATURE_COLS
}

target_min = float("inf")
target_max = float("-inf")

negative_distances = 0
negative_current_delays = 0
negative_targets = 0

date_counts = defaultdict(int)

# ------------------------------------------------------------
# 4. CHUNK-WISE DATA VALIDATION
# ------------------------------------------------------------

print("\nReading dataset in chunks...")

for chunk_no, chunk in enumerate(
    pd.read_csv(
        FEATURE_PATH,
        usecols=REQUIRED_COLS,
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    total_rows += len(chunk)

    # Unique train IDs
    all_trains.update(
        chunk["train"].dropna().astype(str).unique()
    )

    # Missing-value audit
    for col in REQUIRED_COLS:
        missing_counts[col] += chunk[col].isna().sum()

    # Predictor validation
    for col in FEATURE_COLS:

        values = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

        invalid_numeric[col] += values.isna().sum()

        if values.notna().any():

            feature_min[col] = min(
                feature_min[col],
                values.min()
            )

            feature_max[col] = max(
                feature_max[col],
                values.max()
            )

    # Target validation — kept separate from predictors
    target_values = pd.to_numeric(
        chunk[TARGET_COL],
        errors="coerce"
    )

    invalid_numeric[TARGET_COL] += (
        target_values.isna().sum()
    )

    if target_values.notna().any():

        target_min = min(
            target_min,
            target_values.min()
        )

        target_max = max(
            target_max,
            target_values.max()
        )

    # Negative-value checks
    negative_distances += (
        pd.to_numeric(
            chunk["distance_to_next"],
            errors="coerce"
        ) < 0
    ).sum()

    negative_current_delays += (
        pd.to_numeric(
            chunk["current_arr_delay"],
            errors="coerce"
        ) < 0
    ).sum()

    negative_targets += (
        target_values < 0
    ).sum()

    # Date-wise row counts
    dates = pd.to_datetime(
        chunk["date"],
        errors="coerce"
    )

    for date, count in dates.dt.strftime(
        "%Y-%m-%d"
    ).value_counts().items():

        date_counts[date] += int(count)

    if chunk_no % 3 == 0:
        print(
            f"Processed {total_rows:,} rows..."
        )

# ------------------------------------------------------------
# 5. DATASET VALIDATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATASET SUMMARY")
print("=" * 65)

print(f"Total rows: {total_rows:,}")
print(f"Unique trains: {len(all_trains):,}")
print(f"Unique dates: {len(date_counts):,}")

print("\nMissing values:")

for col in REQUIRED_COLS:
    print(
        f"{col}: {missing_counts[col]:,}"
    )

print("\nInvalid numeric values:")

for col in FEATURE_COLS + [TARGET_COL]:
    print(
        f"{col}: {invalid_numeric[col]:,}"
    )

print("\nNegative-value checks:")
print(
    f"Negative distance_to_next: {negative_distances:,}"
)
print(
    f"Negative current_arr_delay: {negative_current_delays:,}"
)
print(
    f"Negative next_arr_delay: {negative_targets:,}"
)

# ------------------------------------------------------------
# 6. FEATURE AND TARGET RANGES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE RANGES")
print("=" * 65)

for col in FEATURE_COLS:

    minimum = feature_min[col]
    maximum = feature_max[col]

    print(
        f"{col:25s} | Min: {minimum:12.4f} "
        f"| Max: {maximum:12.4f}"
    )

print("\nTARGET RANGE")

print(
    f"{TARGET_COL:25s} | Min: {target_min:12.4f} "
    f"| Max: {target_max:12.4f}"
)

# ------------------------------------------------------------
# 7. DATE-WISE DATA DISTRIBUTION
# ------------------------------------------------------------

date_summary = pd.DataFrame(
    list(date_counts.items()),
    columns=["date", "row_count"]
)

date_summary["date"] = pd.to_datetime(
    date_summary["date"]
)

date_summary = date_summary.sort_values(
    "date"
).reset_index(drop=True)

print("\n" + "=" * 65)
print("DATE-WISE DATA DISTRIBUTION")
print("=" * 65)

print(date_summary.to_string(index=False))

# ------------------------------------------------------------
# 8. CHRONOLOGICAL TRAIN-TEST SPLIT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CHRONOLOGICAL SPLIT")
print("=" * 65)

print("Training period: September 1–20, 2024")
print("Testing period : September 21–30, 2024")

train_dates = date_summary[
    date_summary["date"].dt.day <= 20
]

test_dates = date_summary[
    date_summary["date"].dt.day >= 21
]

chron_train_rows = train_dates["row_count"].sum()
chron_test_rows = test_dates["row_count"].sum()

print(
    f"\nTraining rows: {chron_train_rows:,}"
)

print(
    f"Testing rows : {chron_test_rows:,}"
)

print(
    f"Total rows   : {chron_train_rows + chron_test_rows:,}"
)

# ------------------------------------------------------------
# 9. UNSEEN-TRAIN SPLIT MANIFEST
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("UNSEEN-TRAIN SPLIT")
print("=" * 65)

sorted_trains = np.array(
    sorted(all_trains)
)

rng = np.random.default_rng(
    RANDOM_STATE
)

shuffled_indices = rng.permutation(
    len(sorted_trains)
)

test_size = int(
    np.ceil(len(sorted_trains) * 0.20)
)

test_indices = shuffled_indices[:test_size]
train_indices = shuffled_indices[test_size:]

unseen_test_trains = set(
    sorted_trains[test_indices]
)

unseen_train_trains = set(
    sorted_trains[train_indices]
)

assert unseen_train_trains.isdisjoint(
    unseen_test_trains
)

manifest = pd.DataFrame({
    "train": sorted_trains,
    "split": [
        "test" if train in unseen_test_trains
        else "train"
        for train in sorted_trains
    ]
})

manifest_path = os.path.join(
    OUTPUT_DIR,
    "unseen_train_split_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

print(
    f"Training trains: {len(unseen_train_trains):,}"
)

print(
    f"Testing trains : {len(unseen_test_trains):,}"
)

print(
    f"Total trains   : {len(sorted_trains):,}"
)

print(
    f"Train overlap  : "
    f"{len(unseen_train_trains.intersection(unseen_test_trains))}"
)

print(
    f"\nSplit manifest saved: {manifest_path}"
)

# ------------------------------------------------------------
# 10. FINAL VALIDATION STATUS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION STATUS")
print("=" * 65)

print("Total dataset rows:", f"{total_rows:,}")
print("Predictor count:", len(FEATURE_COLS))
print("Target:", TARGET_COL)

print(
    "Missing values:",
    sum(missing_counts.values())
)

print(
    "Invalid numeric values:",
    sum(invalid_numeric.values())
)

print(
    "Chronological split:",
    f"{chron_train_rows:,} train / {chron_test_rows:,} test"
)

print(
    "Unseen-train split:",
    f"{len(unseen_train_trains):,} train / "
    f"{len(unseen_test_trains):,} test trains"
)

print("\nValidation cell completed.")
print("Review the output before proceeding.")


RAILPULSE AI — DATA VALIDATION

Dataset: ../data/processed/railpulse_model_features.csv
Predictor columns: 9
Target column: next_arr_delay

Reading dataset in chunks...
Processed 300,000 rows...
Processed 600,000 rows...
Processed 900,000 rows...
Processed 1,101,443 rows...

DATASET SUMMARY
Total rows: 1,101,443
Unique trains: 3,146
Unique dates: 30

Missing values:
train: 0
date: 0
current_station: 0
next_station: 0
current_arr_delay: 0
distance_to_next: 0
current_distance: 0
current_sequence: 0
arrival_hour_sin: 0
arrival_hour_cos: 0
day_of_week_sin: 0
day_of_week_cos: 0
is_weekend: 0
next_arr_delay: 0

Invalid numeric values:
current_arr_delay: 0
distance_to_next: 0
current_distance: 0
current_sequence: 0
arrival_hour_sin: 0
arrival_hour_cos: 0
day_of_week_sin: 0
day_of_week_cos: 0
is_weekend: 0
next_arr_delay: 0

Negative-value checks:
Negative distance_to_next: 0
Negative current_arr_delay: 0
Negative next_arr_delay: 0

FEATURE RANGES
current_arr_delay         | Min:       0.0000 

In [1]:

import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# Project paths
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed" / "railpulse_model_features.csv"
MANIFEST = ROOT / "outputs" / "tables" / "three_way_train_split_manifest.csv"
OUT = ROOT / "outputs" / "tables"

# Final four selected features
features = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence"
]

columns = ["train", "next_arr_delay"] + features

# Read the train-ID split
manifest = pd.read_csv(MANIFEST, dtype={"train": str})
split_map = dict(zip(manifest["train"], manifest["split"]))

# Read only the columns needed
df = pd.read_csv(DATA, usecols=columns, dtype={"train": str})
df["split"] = df["train"].map(split_map)

# Separate internal training and validation data
train_df = df[df["split"] == "internal_train"].copy()
val_df = df[df["split"] == "validation"].copy()

# Target = change in delay
train_y = train_df["next_arr_delay"] - train_df["current_arr_delay"]
val_y = val_df["next_arr_delay"] - val_df["current_arr_delay"]

X_train = train_df[features]
X_val = val_df[features]

# Train selected model
model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.06,
    max_iter=100,
    max_leaf_nodes=31,
    min_samples_leaf=40,
    l2_regularization=2,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=12,
    random_state=42
)

model.fit(X_train, train_y)

# Use a reproducible validation sample for importance
sample_size = min(30000, len(X_val))
X_sample = X_val.sample(n=sample_size, random_state=42)
y_sample = val_y.loc[X_sample.index]

result = permutation_importance(
    model,
    X_sample,
    y_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=1
)

importance = pd.DataFrame({
    "Feature": features,
    "Importance_Mean": result.importances_mean,
    "Importance_Std": result.importances_std
}).sort_values("Importance_Mean", ascending=False)

OUT.mkdir(parents=True, exist_ok=True)
importance.to_csv(
    OUT / "final_model_permutation_importance.csv",
    index=False
)

print("FINAL FOUR-FEATURE MODEL IMPORTANCE")
print("Training rows:", len(train_df))
print("Validation sample:", len(X_sample))
print(importance.to_string(index=False))
print("\nSaved successfully.")


FINAL FOUR-FEATURE MODEL IMPORTANCE
Training rows: 699293
Validation sample: 30000
          Feature  Importance_Mean  Importance_Std
current_arr_delay         0.306696        0.001685
 distance_to_next         0.175345        0.007047
 current_distance         0.153002        0.003396
 current_sequence         0.114790        0.003526

Saved successfully.
